# SC4002 - Lab Assigment

## Import and Configs

In [1]:
import os
import re
import math
import copy
import numpy as np
import pandas as pd
import random
import time
import matplotlib.pyplot as plt

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gensim.downloader as api

import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

True

In [2]:
DATA_FOLDER = r"data"
MODEL_FOLDER = r"model"

os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(MODEL_FOLDER, exist_ok=True)

DATASET_FILE = "en-US.jsonl"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Fixing seed to remove randomness, for better comparisons
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

In [4]:
# Pretraining Settings
RUN_PRETRAINING = True 
PRETRAINING_TASK = "Joint" # "Reconstruction" / "Joint", 
MASK_RATIO = 0.15 # 0 for no masking, but reconstruction have to use masking
LABEL_SMOOTHING = 0.1 

# Final Model Settings
USE_BACKBONE = True # True to use pretraining, False to bypass the backbone layers 
RUN_INTENT_TRAINING = True 
RUN_SLOT_TRAINING = True

In [5]:
# Name checkpoints after the setup that produced them so runs do not overwrite each other
BACKBONE_TAG = PRETRAINING_TASK.lower() + ("_masked" if MASK_RATIO > 0 else "")
HEAD_TAG = BACKBONE_TAG if USE_BACKBONE else "no_backbone"

UNTRAINED_BACKBONE_PATH = os.path.join(MODEL_FOLDER, "untrained_backbone.pt")
BEST_BACKBONE_PATH = os.path.join(MODEL_FOLDER, f"best_backbone_{BACKBONE_TAG}.pt")
BEST_INTENT_PATH = os.path.join(MODEL_FOLDER, f"best_intent_{HEAD_TAG}.pt")
BEST_SLOT_PATH = os.path.join(MODEL_FOLDER, f"best_slot_{HEAD_TAG}.pt")

## Data Loading

In [6]:
df = pd.read_json(os.path.join(DATA_FOLDER,DATASET_FILE), lines=True)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst example:")
print(df.iloc[0])

Dataset shape: (16521, 8)

Columns:
['id', 'locale', 'partition', 'scenario', 'intent', 'utt', 'annot_utt', 'worker_id']

First example:
id                                                           0
locale                                                   en-US
partition                                                 test
scenario                                                 alarm
intent                                               alarm_set
utt                            wake me up at five am this week
annot_utt    wake me up at [time : five am] [date : this week]
worker_id                                                    1
Name: 0, dtype: object


In [7]:
# Check the number of examples in each partition
print("Partition distribution:")
print(df["partition"].value_counts())

# Check the number of unique intents
print("\nNumber of unique intents:")
print(df["intent"].nunique())

# Check the number of examples per intent
print("\nExamples per intent:")
display(df["intent"].value_counts().describe())

# Check for missing values
print("\nMissing values:")
print(df[["utt", "annot_utt", "intent", "partition"]].isnull().sum())

# Check whether the dataset contains only en-US
print("\nLocales:")
print(df["locale"].value_counts())

Partition distribution:
partition
train    11514
test      2974
dev       2033
Name: count, dtype: int64

Number of unique intents:
60

Examples per intent:


count      60.000000
mean      275.350000
std       253.817352
min         6.000000
25%       114.750000
50%       183.000000
75%       315.500000
max      1150.000000
Name: count, dtype: float64


Missing values:
utt          0
annot_utt    0
intent       0
partition    0
dtype: int64

Locales:
locale
en-US    16521
Name: count, dtype: int64


In [8]:
train_df = df[df["partition"] == "train"].copy()
val_df = df[df["partition"] == "dev"].copy()
test_df = df[df["partition"] == "test"].copy()

In [9]:
x_train = train_df["utt"].tolist()
y_train = train_df["intent"].tolist()
z_train = train_df["annot_utt"].tolist()

x_val = val_df["utt"].tolist()
y_val = val_df["intent"].tolist()
z_val = val_df["annot_utt"].tolist()

x_test = test_df["utt"].tolist()
y_test = test_df["intent"].tolist()
z_test = test_df["annot_utt"].tolist()

In [10]:
print("Training examples:", len(x_train))
print("Validation examples:", len(x_val))
print("Test examples:", len(x_test))

print("\nExample utterance:", x_train[0])
print("Example intent:", y_train[0])
print("Example annotated utterance:", z_train[0])

Training examples: 11514
Validation examples: 2033
Test examples: 2974

Example utterance: wake me up at nine am on friday
Example intent: alarm_set
Example annotated utterance: wake me up at [time : nine am] on [date : friday]


In [ ]:
glove = api.load("glove-wiki-gigaword-300")

In [ ]:
# Check vocabulary size
print("Vocabulary size:", len(glove.index_to_key))

In [ ]:
token_pattern = re.compile(r'\w+|[^\w\s]')

def tokenize_with_offsets(text):
    matches = token_pattern.finditer(text)
    return [(m.group().lower(), m.start(), m.end()) for m in matches]

def tokenize(text):
    return [token for token, _, _ in tokenize_with_offsets(text)]

In [ ]:
def check_glove_coverage(texts, glove):
    all_tokens = [
        token
        for text in texts
        for token in tokenize(text)
    ]

    unique_tokens = set(all_tokens)
    known_tokens = {token for token in unique_tokens if token in glove}

    total_tokens = len(all_tokens)
    total_unique = len(unique_tokens)
    known_count = sum(token in glove for token in all_tokens)

    print(f"Total tokens: {total_tokens:,}")
    print(f"Unique words: {total_unique:,}")
    print(f"Unique words covered: {len(known_tokens):,}")
    print(f"Unique words OOV: {total_unique - len(known_tokens):,}")

    print(f"\nUnique-word coverage: {len(known_tokens) / total_unique * 100:.2f}%")
    print(f"Token coverage: {known_count / total_tokens * 100:.2f}%")

    oov_words = sorted(unique_tokens - known_tokens)
    print("\nExample OOV words:")
    print(oov_words[:30])

In [ ]:
# Check each split
print("TRAINING SET")
check_glove_coverage(x_train, glove)

print("\nVALIDATION SET")
check_glove_coverage(x_val, glove)

print("\nTEST SET")
check_glove_coverage(x_test, glove)

In [ ]:
# Special token IDs
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
MASK_TOKEN = "<MASK>"

PAD_ID = 0
UNK_ID = 1
MASK_ID = 2

# Tokenize the training utterances
tokenized_x_train = [tokenize(text) for text in x_train]

# Build vocabulary using the training set
vocab_words = sorted({
    token
    for tokens in tokenized_x_train
    for token in tokens
})

# Create word-to-ID mapping
word_to_id = {
    PAD_TOKEN: PAD_ID,
    UNK_TOKEN: UNK_ID,
    MASK_TOKEN: MASK_ID
}

for word in vocab_words:
    if word not in word_to_id:
        word_to_id[word] = len(word_to_id)

# Create reverse mapping
id_to_word = {idx: word for word, idx in word_to_id.items()}

print("Training vocabulary size (incl. PAD/UNK/MASK):", len(word_to_id))
print("Training tokens missing from GloVe:", sum(word not in glove for word in vocab_words))

In [ ]:
USE_LEMMA_FALLBACK = False

lemmatizer = WordNetLemmatizer()

def lemmatize_token(token):
    # Try the noun form first (plurals), then the verb form (tenses)
    lemma = lemmatizer.lemmatize(token, pos='n')
    if lemma == token:
        lemma = lemmatizer.lemmatize(token, pos='v')
    return lemma

def token_to_id(token, word_to_id, use_lemma):
    # Known words keep their own ID; only unseen words fall back to their lemma
    if token in word_to_id:
        return word_to_id[token]
    if use_lemma:
        lemma = lemmatize_token(token)
        if lemma in word_to_id:
            return word_to_id[lemma]
    return UNK_ID

In [ ]:
embedding_dim = glove.vector_size
vocab_size = len(word_to_id)

rng = np.random.default_rng(SEED)

embedding_matrix = rng.normal(
    0, 0.1, (vocab_size, embedding_dim)
).astype(np.float32)

embedding_matrix[PAD_ID] = 0

for word, idx in word_to_id.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]

print("Embedding matrix shape:", embedding_matrix.shape)
print("Padding vector:", embedding_matrix[PAD_ID])
print("Unknown vector shape:", embedding_matrix[UNK_ID].shape)
print("Words missing from GloVe:",sum(word not in glove for word in vocab_words))

In [ ]:
max_train_len = max(len(tokens) for tokens in tokenized_x_train)
max_len = max(64, max_train_len)
print("Longest training sentence:", max_train_len)
print("Padding length:", max_len)

In [ ]:
intents = sorted(set(y_train))

intent_to_id = {
    intent: idx
    for idx, intent in enumerate(intents)
}

id_to_intent = {
    idx: intent
    for intent, idx in intent_to_id.items()
}

num_classes = len(intent_to_id)
print("Number of intent classes:", num_classes)

In [ ]:
slot_pattern = re.compile(r'\[\s*([^:\[\]]+?)\s*:\s*([^\[\]]*?)\]')
SLOT_PAD_ID = -100  

def parse_annotated_utterance(utt, annot_utt):
    spans = []
    plain_text = ''
    last_end = 0

    for match in slot_pattern.finditer(annot_utt):
        plain_text += annot_utt[last_end:match.start()]
        slot_type = match.group(1).strip()
        slot_value = match.group(2)
        start = len(plain_text)
        plain_text += slot_value
        spans.append({
            'type': slot_type,
            'start': start,
            'end': len(plain_text),
            'text': slot_value
        })
        last_end = match.end()

    plain_text += annot_utt[last_end:]
    if plain_text != utt:
        raise ValueError(f"Slot annotation does not match utterance: {utt!r}")

    return spans


In [ ]:
def make_aligned_tokens(utt, spans):
    tags = []
    offsets = []
    prev_slot = None

    for word, start, end in tokenize_with_offsets(utt):
        current_slot = None
        for i, span in enumerate(spans):
            if start < span['end'] and end > span['start']:
                if start < span['start'] or end > span['end']:
                    raise ValueError(f"A token cuts across a slot boundary: {utt!r}")
                if current_slot is not None:
                    raise ValueError(f"A token overlaps two slots: {utt!r}")
                current_slot = i

        if current_slot is None:
            tags.append('O')
        else:
            prefix = 'I' if current_slot == prev_slot else 'B'
            tags.append(f"{prefix}-{spans[current_slot]['type']}")

        offsets.append((start, end))
        prev_slot = current_slot

    for span in spans:
        if not any(s >= span['start'] and e <= span['end'] for s, e in offsets):
            raise ValueError(f"Could not align slot {span['text']!r} in {utt!r}")

    return tags, offsets


In [ ]:
def prepare_joint_records(split_df):
    records = []
    
    for row_index, row in split_df.iterrows():
        text = row['utt']
        spans = parse_annotated_utterance(text, row['annot_utt'])
        tags, offsets = make_aligned_tokens(text, spans)
        tokens = tokenize(text)

        if len(tokens) != len(tags):
            raise ValueError(f"Token and slot lengths differ at row {row_index}")

        records.append({
            'source_row': int(row_index),
            'utt': text,
            'tokens': tokens,
            'offsets': offsets,
            'gold_spans': spans,
            'slot_tags': tags,
            'intent': row['intent']
        })
    return records


In [ ]:
train_records = prepare_joint_records(train_df)
val_records = prepare_joint_records(val_df)
test_records = prepare_joint_records(test_df)

training_slot_types = sorted({span['type'] for row in train_records for span in row['gold_spans']})
slot_tags = ['O']

for slot_type in training_slot_types:
    slot_tags.extend([f'B-{slot_type}', f'I-{slot_type}'])

slot_to_id = {tag: i for i, tag in enumerate(slot_tags)}
id_to_slot = {i: tag for tag, i in slot_to_id.items()}
num_slot_labels = len(slot_to_id)

print('Joint dataset sizes:', len(train_records), len(val_records), len(test_records))
print('Slot types in training:', len(training_slot_types))
print('BIO labels:', num_slot_labels)
print('Example:', train_records[0]['slot_tags'])

In [ ]:
class JointNLUDataset(Dataset):
    def __init__(self, records, word_to_id, intent_to_id, 
                 slot_to_id, max_len, use_lemma=False):
        self.records = records
        self.input_ids = []
        self.attention_masks = []
        self.intent_ids = []
        self.slot_ids = []

        for row in records:
            words = [token_to_id(word, word_to_id, use_lemma) for word in row['tokens']]
            slots = [slot_to_id[tag] for tag in row['slot_tags']]
            if len(words) != len(slots):
                raise ValueError('Token and slot label counts do not match')
            if len(words) > max_len:
                raise ValueError(f"Sentence has {len(words)} tokens, limit is {max_len}")

            padding = max_len - len(words)
            self.input_ids.append(torch.tensor(words + [PAD_ID] * padding))
            self.attention_masks.append(torch.tensor([1] * len(words) + [0] * padding, 
                                                     dtype=torch.bool))
            self.intent_ids.append(torch.tensor(intent_to_id[row['intent']]))
            self.slot_ids.append(torch.tensor(slots + [SLOT_PAD_ID] * padding))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        return {
            'input_ids': self.input_ids[index],
            'attention_mask': self.attention_masks[index],
            'intent_labels': self.intent_ids[index],
            'slot_labels': self.slot_ids[index],
            'utt': row['utt'],
            'offsets': row['offsets'],
            'gold_spans': row['gold_spans'],
            'source_row': row['source_row']
        }

In [ ]:
def joint_collate_fn(samples):
    batch = {}
    for key in ('input_ids', 'attention_mask', 'intent_labels', 'slot_labels'):
        batch[key] = torch.stack([item[key] for item in samples])
    for key in ('utt', 'offsets', 'gold_spans', 'source_row'):
        batch[key] = [item[key] for item in samples]
    return batch

In [ ]:
train_joint_dataset = JointNLUDataset(train_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)
val_joint_dataset = JointNLUDataset(val_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)
test_joint_dataset = JointNLUDataset(test_records, word_to_id, intent_to_id, slot_to_id, max_len, use_lemma=USE_LEMMA_FALLBACK)

In [ ]:
batch_size = 64

train_joint_loader = DataLoader(train_joint_dataset, 
                                batch_size=batch_size, 
                                shuffle=True,
                                collate_fn=joint_collate_fn,
                                generator=torch.Generator().manual_seed(SEED))

val_joint_loader = DataLoader(val_joint_dataset, 
                              batch_size=batch_size, 
                              shuffle=False,
                              collate_fn=joint_collate_fn)

test_joint_loader = DataLoader(test_joint_dataset, 
                               batch_size=batch_size, 
                               shuffle=False,
                               collate_fn=joint_collate_fn)

batch = next(iter(train_joint_loader))
print('Input IDs:', batch['input_ids'].shape)
print('Intent labels:', batch['intent_labels'].shape)
print('Slot labels:', batch['slot_labels'].shape)
print('Attention mask:', batch['attention_mask'].shape)

In [ ]:
def build_vocab(records):
    vocab = {PAD_TOKEN: PAD_ID, UNK_TOKEN: UNK_ID, MASK_TOKEN: MASK_ID}
    for word in sorted({t for row in records for t in row['tokens']}):
        vocab[word] = len(vocab)
    return vocab

def build_embedding_matrix(vocab):
    matrix_rng = np.random.default_rng(SEED)
    matrix = matrix_rng.normal(0, 0.1, (len(vocab), embedding_dim)).astype(np.float32)
    matrix[PAD_ID] = 0
    for word, idx in vocab.items():
        if word in glove:
            matrix[idx] = glove[word]
    return matrix

def make_loader(records, vocab, use_lemma=False, shuffle=False):
    dataset = JointNLUDataset(records, vocab, intent_to_id, slot_to_id, max_len, use_lemma=use_lemma)
    generator = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=joint_collate_fn, generator=generator)

def lemmatize_records(records):
    return [{**row, 'tokens': [lemmatize_token(t) for t in row['tokens']]} for row in records]

## Model Architecture

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128, scale=1.0):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))
        self.scale = scale

    def forward(self, x):
        return x + self.scale * self.pe[:, :x.size(1), :]

In [ ]:
class BackboneEncoder(nn.Module):
    def __init__(
        self,
        embedding_matrix,
        pad_id,
        max_len=128,
        num_layers=4,
        num_heads=4,
        dim_feedforward=1024,
        dropout=0.1,
        pe_scale=0.5,
    ):
        super().__init__()

        embedding_matrix = torch.as_tensor(
            embedding_matrix,
            dtype=torch.float32,
        )

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True,
            padding_idx=pad_id,
        )

        d_model = embedding_matrix.size(1)

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=max_len,
            scale=pe_scale,
        )

        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        self.pad_id = pad_id
        self.d_model = d_model

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x = self.positional_encoding(x)
        x = self.dropout(x)

        padding_mask = ~attention_mask.bool()

        x = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask,
        )

        return x

In [ ]:
class BackboneReconstructor(nn.Module):
    def __init__(
        self,
        d_model,
        vocab_size,
        num_layers=2,
        num_heads=4,
        dim_feedforward=1024,
        dropout=0.1,
    ):
        super().__init__()

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            decoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )

        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
        )

    def forward(self, hidden_states, attention_mask):
        padding_mask = ~attention_mask.bool()
        x = self.transformer(
            hidden_states,
            src_key_padding_mask=padding_mask,
        )
        logits = self.output_projection(x)

        return logits

In [ ]:
class BackboneJointHead(nn.Module):
    def __init__(
        self,
        d_model,
        num_intents,
        num_slot_labels,
        dropout=0.1,
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout)
        self.intent_classifier = nn.Linear(d_model, num_intents)
        self.slot_classifier = nn.Linear(d_model, num_slot_labels)

    def forward(self, hidden_states, attention_mask):
        valid_tokens = attention_mask.unsqueeze(-1).to(
            hidden_states.dtype
        )

        pooled = (hidden_states * valid_tokens).sum(dim=1)
        pooled = pooled / valid_tokens.sum(dim=1).clamp(min=1.0)

        intent_logits = self.intent_classifier(self.dropout(pooled))
        slot_logits = self.slot_classifier(self.dropout(hidden_states))

        return intent_logits, slot_logits

In [ ]:
class Backbone(nn.Module):
    def __init__(self, encoder, task_head, task):
        super().__init__()

        self.encoder = encoder
        self.task_head = task_head
        self.task = task

    def forward(
        self,
        input_ids,
        attention_mask,
        labels=None,
        intent_labels=None,
        slot_labels=None,
    ):
        hidden_states = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        if self.task == "Reconstruction":
            logits = self.task_head(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
            )

            loss = None

            if labels is not None:
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)),
                    labels.reshape(-1),
                    ignore_index=-100,
                )

            return {
                "loss": loss,
                "logits": logits,
            }

        intent_logits, slot_logits = self.task_head(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
        )

        loss = None

        if intent_labels is not None and slot_labels is not None:
            loss = F.cross_entropy(
                intent_logits,
                intent_labels,
                label_smoothing=LABEL_SMOOTHING,
            )

            if (slot_labels != SLOT_PAD_ID).any():
                loss = loss + F.cross_entropy(
                    slot_logits.reshape(-1, slot_logits.size(-1)),
                    slot_labels.reshape(-1),
                    ignore_index=SLOT_PAD_ID,
                )

        return {
            "loss": loss,
            "logits": (intent_logits, slot_logits),
        }

In [ ]:
class IntentClassifier(nn.Module):
    def __init__(
        self,
        backbone_encoder,
        num_classes,
        num_layers=2,
        nhead=4,
        dim_feedforward=1024,
        dropout=0.1,
        pad_id=PAD_ID,
        use_backbone_layers=True,
    ):
        super().__init__()

        self.backbone_encoder = backbone_encoder
        self.pad_id = pad_id
        self.use_backbone_layers = use_backbone_layers

        # Freeze backbone
        for param in self.backbone_encoder.parameters():
            param.requires_grad = False

        d_model = self.backbone_encoder.d_model

        if d_model % nhead != 0:
            raise ValueError(
                f"d_model={d_model} must be divisible by nhead={nhead}"
            )

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.intent_encoder = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        padding_mask = ~attention_mask.bool()

        with torch.no_grad():
            if self.use_backbone_layers:
                hidden_states = self.backbone_encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
            else:
                hidden_states = self.backbone_encoder.positional_encoding(
                    self.backbone_encoder.embedding(input_ids)
                )

        hidden_states = self.intent_encoder(
            hidden_states,
            src_key_padding_mask=padding_mask,
        )

        valid_tokens = attention_mask.unsqueeze(-1).to(
            hidden_states.dtype
        )

        pooled = (hidden_states * valid_tokens).sum(dim=1)
        pooled = pooled / valid_tokens.sum(dim=1).clamp(min=1.0)

        logits = self.classifier(self.dropout(pooled))

        return logits

In [ ]:
class SlotFiller(nn.Module):
    def __init__(
        self,
        backbone_encoder,
        num_slot_labels,
        num_layers=4,
        nhead=4,
        dim_feedforward=1024,
        dropout=0.3,
        use_backbone_layers=True,
    ):
        super().__init__()

        self.backbone_encoder = backbone_encoder
        self.use_backbone_layers = use_backbone_layers

        # Freeze the pretrained or randomly initialized backbone
        for param in self.backbone_encoder.parameters():
            param.requires_grad = False

        d_model = self.backbone_encoder.d_model

        # Trainable Transformer for slot extraction
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.slot_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.dropout = nn.Dropout(dropout)

        # One classification per token
        self.classifier = nn.Linear(
            d_model,
            num_slot_labels,
        )

    def forward(self, input_ids, attention_mask):
        padding_mask = ~attention_mask.bool()

        # Frozen backbone features
        self.backbone_encoder.eval()

        with torch.no_grad():
            if self.use_backbone_layers:
                hidden_states = self.backbone_encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
            else:
                embeddings = self.backbone_encoder.embedding(
                    input_ids
                )
                hidden_states = (
                    self.backbone_encoder.positional_encoding(
                        embeddings
                    )
                )

        # Trainable contextual representations
        hidden_states = self.slot_encoder(
            hidden_states,
            src_key_padding_mask=padding_mask,
        )

        # Token-level BIO logits
        logits = self.classifier(
            self.dropout(hidden_states)
        )

        return logits

## Pretraining Loop

The backbone is pretrained with `PRETRAINING_TASK`:
- **Reconstruction**: predict the tokens replaced by `<MASK>` (every token when `MASK_RATIO = 0`).
- **Joint**: intent cross-entropy (label smoothing `LABEL_SMOOTHING`) + slot token cross-entropy, as in the BiLSTM joint loss. Masked tokens are ignored in the slot loss.

In [ ]:
# Backbone pretraining config
PRETRAINING_EPOCH = 500    # max epochs
PRETRAINING_PATIENCE = 10  
PRETRAINING_REDUCELR_PATIENCE = 3 

In [ ]:
encoder = BackboneEncoder(
    embedding_matrix=embedding_matrix,
    pad_id=PAD_ID,
    max_len=128,
    num_layers=5,
    num_heads=4,
    dim_feedforward=1024,
    dropout=0.1,
)

if PRETRAINING_TASK == "Reconstruction":
    task_head = BackboneReconstructor(
        d_model=encoder.d_model,
        vocab_size=embedding_matrix.shape[0],
        num_layers=2,
        num_heads=4,
        dim_feedforward=1024,
        dropout=0.1,
    )
else:
    task_head = BackboneJointHead(
        d_model=encoder.d_model,
        num_intents=num_classes,
        num_slot_labels=num_slot_labels,
        dropout=0.1,
    )

In [ ]:
backbone_model = Backbone(
    encoder=encoder,
    task_head=task_head,
    task=PRETRAINING_TASK,
)

In [ ]:
best_loss = float("inf")
epochs_without_improvement = 0

train_loss_history = []

In [ ]:
def create_pretraining_inputs(input_ids, attention_mask, mask_ratio=MASK_RATIO, mask_token_id=MASK_ID):
    masked_input_ids = input_ids.clone()

    special_mask = input_ids.eq(PAD_ID)
    eligible_tokens = attention_mask.bool() & ~special_mask

    num_eligible = eligible_tokens.sum(dim=1, keepdim=True)
    num_to_mask = torch.floor(num_eligible.float() * mask_ratio + 1e-6).long()

    scores = torch.rand(input_ids.shape, device=input_ids.device)
    scores = scores.masked_fill(~eligible_tokens, float("inf"))
    ranks = scores.argsort(dim=1).argsort(dim=1)

    masked_positions = (ranks < num_to_mask) & eligible_tokens

    masked_input_ids[masked_positions] = mask_token_id

    return masked_input_ids, masked_positions

In [ ]:
backbone_model = backbone_model.to(device)

pretraining_optimizer = optim.AdamW(
    (p for p in backbone_model.parameters() if p.requires_grad),
    lr=1e-4,
    weight_decay=0.01,
)

pretraining_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    pretraining_optimizer,
    mode="min",
    factor=0.1,
    patience=PRETRAINING_REDUCELR_PATIENCE,
    min_lr=1e-6,
)

In [ ]:
def train_pretraining_epoch(
    model,
    dataloader,
    optimizer,
    device,
    mask_ratio=MASK_RATIO,
    mask_token_id=MASK_ID,
):
    model.train()

    total_loss = 0.0
    total_batches = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        masked_input_ids, masked_positions = create_pretraining_inputs(
            input_ids=input_ids,
            attention_mask=attention_mask,
            mask_ratio=mask_ratio,
            mask_token_id=mask_token_id,
        )

        if model.task == "Reconstruction":
            labels = input_ids.clone()

            if mask_ratio > 0:
                labels[~masked_positions] = -100
            else:
                labels[~attention_mask.bool()] = -100

            # skip batch if no masked tokens
            if not (labels != -100).any():
                continue

            loss_inputs = {"labels": labels}
        else:
            slot_labels = batch["slot_labels"].to(device).clone()

            slot_labels[masked_positions] = SLOT_PAD_ID

            loss_inputs = {
                "intent_labels": batch["intent_labels"].to(device).long(),
                "slot_labels": slot_labels,
            }

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=masked_input_ids,
            attention_mask=attention_mask,
            **loss_inputs,
        )

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1
        
    return total_loss / total_batches

In [ ]:
if RUN_PRETRAINING:
    torch.save(
        {
            k: v.detach().cpu().clone()
            for k, v in backbone_model.encoder.state_dict().items()
        },
        UNTRAINED_BACKBONE_PATH,
    )

    for epoch in range(PRETRAINING_EPOCH):
        train_loss = train_pretraining_epoch(
            model=backbone_model,
            dataloader=train_joint_loader,
            optimizer=pretraining_optimizer,
            device=device,
            mask_ratio=MASK_RATIO,
            mask_token_id=MASK_ID,
        )

        train_loss_history.append(train_loss)

        pretraining_scheduler.step(train_loss)

        if train_loss < best_loss:
            best_loss = train_loss
            epochs_without_improvement = 0

            best_model_state = {
                k: v.detach().cpu().clone()
                for k, v in backbone_model.state_dict().items()
            }

            torch.save(
                best_model_state,
                BEST_BACKBONE_PATH,
            )
        else:
            epochs_without_improvement += 1

        current_lr = pretraining_optimizer.param_groups[0]["lr"]

        print(
            f"Epoch [{epoch + 1}/{PRETRAINING_EPOCH}] "
            f"| Train {PRETRAINING_TASK} Loss: {train_loss:.4f} "
            f"| Best Loss: {best_loss:.4f} "
            f"| LR: {current_lr:.2e}"
        )

        if epochs_without_improvement >= PRETRAINING_PATIENCE:
            print(f"Early stopping triggered at epoch {epoch + 1}.")
            break

    backbone_model.load_state_dict(best_model_state)

    print(f"Best {PRETRAINING_TASK} Loss: {best_loss:.4f}")
else:
    print("Skipping backbone pretraining.")

    if not os.path.exists(UNTRAINED_BACKBONE_PATH):
        torch.save(
            {
                k: v.detach().cpu().clone()
                for k, v in backbone_model.encoder.state_dict().items()
            },
            UNTRAINED_BACKBONE_PATH,
        )

## Training Loop (Intent Classifier)

In [ ]:
# Intent classifier training config
INTENT_EPOCH = 500    # max epochs
INTENT_PATIENCE = 20 
INTENT_REDUCELR_PATIENCE = 10

In [ ]:
backbone_path = BEST_BACKBONE_PATH if USE_BACKBONE else UNTRAINED_BACKBONE_PATH

if not os.path.exists(backbone_path):
    raise FileNotFoundError(
        f"{backbone_path} not found."
    )

checkpoint = torch.load(
    backbone_path,
    map_location=device,
    weights_only=True,
)

if USE_BACKBONE:
    backbone_model.load_state_dict(checkpoint)
    print(f"Pretrained backbone loaded.")
else:
    backbone_model.encoder.load_state_dict(checkpoint)
    print("Untrained backbone loaded (backbone layers bypassed).")

In [ ]:
torch.manual_seed(SEED)
train_joint_loader.generator.manual_seed(SEED)

intent_model = IntentClassifier(
    backbone_encoder=backbone_model.encoder,
    num_classes=num_classes,
    num_layers=4,
    nhead=4,
    dim_feedforward=1024,
    dropout=0.3,
    pad_id=PAD_ID,
    use_backbone_layers=USE_BACKBONE,
).to(device)

In [ ]:
# Freeze backbone
intent_model.backbone_encoder.eval()

criterion = nn.CrossEntropyLoss()

intent_optimizer = torch.optim.AdamW(
    [
        *intent_model.intent_encoder.parameters(),
        *intent_model.classifier.parameters(),
    ],
    lr=1e-4,
    weight_decay=1e-4,
)

intent_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    intent_optimizer,
    mode="max",
    factor=0.1,
    patience=INTENT_REDUCELR_PATIENCE,
    min_lr=1e-6,
)

# Early stopping flags
best_val_acc = 0.0
epochs_without_improvement = 0

In [ ]:
def train_intent_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
):
    model.train()
    model.backbone_encoder.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in train_loader:
        x_batch = batch["input_ids"].to(device)
        mask_batch = batch["attention_mask"].to(device)
        y_batch = batch["intent_labels"].to(device).long()

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids=x_batch,
            attention_mask=mask_batch,
        )

        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += batch_size

    avg_loss = total_loss / max(total, 1)
    accuracy = correct / max(total, 1)

    return avg_loss, accuracy

In [ ]:
@torch.no_grad()
def evaluate_intent(
    model,
    dataloader,
    criterion,
    device,
):
    model.eval()

    # Freeze backbone
    model.backbone_encoder.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["intent_labels"].to(device).long()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return {
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
    }

In [ ]:
if RUN_INTENT_TRAINING:
    best_val_acc = 0.0

    for epoch in range(INTENT_EPOCH):
        train_loss, train_acc = train_intent_epoch(
            intent_model,
            train_joint_loader,
            criterion,
            intent_optimizer,
            device,
        )

        val_metrics = evaluate_intent(
            intent_model,
            val_joint_loader,
            criterion,
            device,
        )

        val_loss = val_metrics["loss"]
        val_acc = val_metrics["accuracy"]

        old_lr = intent_optimizer.param_groups[0]["lr"]
        intent_scheduler.step(val_acc)
        new_lr = intent_optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1}/{INTENT_EPOCH} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"LR: {new_lr:.2e}"
        )

        if new_lr < old_lr:
            print(f"Learning rate reduced: {old_lr:.2e} -> {new_lr:.2e}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0

            torch.save(
                intent_model.state_dict(),
                BEST_INTENT_PATH,
            )

            print("Saved best intent classifier.")

        else:
            epochs_without_improvement += 1
            print(
                f"No validation accuracy improvement: "
                f"{epochs_without_improvement}/{INTENT_PATIENCE}"
            )

        if epochs_without_improvement >= INTENT_PATIENCE:
            print("Early stopping triggered.")
            break
else:
    print("Skipping intent classifier training.")

In [ ]:
if not os.path.exists(BEST_INTENT_PATH):
    raise FileNotFoundError(
        f"{BEST_INTENT_PATH} not found."
    )

checkpoint = torch.load(
    BEST_INTENT_PATH,
    map_location=device,
    weights_only=True,
)

intent_model.load_state_dict(checkpoint)
intent_model.eval()

print("Best intent classifier loaded.")

In [ ]:
eval_metrics = evaluate_intent(
        intent_model,
        val_joint_loader,
        criterion,
        device,
    )

print(f"Val Acc: {eval_metrics['accuracy']:.4f}")

eval_metrics = evaluate_intent(
        intent_model,
        test_joint_loader,
        criterion,
        device,
    )

print(f"Test Acc: {eval_metrics['accuracy']:.4f}")

## Training Loop (Slot Filling)

In [ ]:
# Slot filling training config
SLOT_EPOCH = 500    # max epochs
SLOT_PATIENCE = 20  
SLOT_REDUCELR_PATIENCE = 10  

In [ ]:
torch.manual_seed(SEED)
train_joint_loader.generator.manual_seed(SEED)

slot_model = SlotFiller(
    backbone_encoder=backbone_model.encoder,
    num_slot_labels=num_slot_labels,
    num_layers=4,
    nhead=4,
    dim_feedforward=1024,
    dropout=0.3,
    use_backbone_layers=USE_BACKBONE,
).to(device)

In [ ]:
# Slot classification loss
slot_criterion = nn.CrossEntropyLoss(
    ignore_index=SLOT_PAD_ID
)

# Only optimize trainable parameters
slot_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        slot_model.parameters()
    ),
    lr=1e-4,
    weight_decay=1e-4,
)

# Reduce learning rate when validation slot F1 stops improving.
slot_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    slot_optimizer,
    mode="max",
    factor=0.1,
    patience=SLOT_REDUCELR_PATIENCE,
    min_lr=1e-6,
)

print("Slot checkpoint:", BEST_SLOT_PATH)
print("Padding label:", SLOT_PAD_ID)

In [ ]:
slot_model.eval()

batch = next(iter(train_joint_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
slot_labels = batch["slot_labels"].to(device)

with torch.no_grad():
    logits = slot_model(input_ids, attention_mask)

    loss = slot_criterion(
        logits.reshape(-1, num_slot_labels),
        slot_labels.reshape(-1)
    )

print("Slot loss:", loss.item())
print("Loss is finite:", torch.isfinite(loss).item())
print("Valid tokens:", (slot_labels != SLOT_PAD_ID).sum().item())

In [ ]:
def decode_slot_spans(pred_ids, offsets, utt, id_to_slot):
    spans = []
    current_type = None
    start = None
    end = None

    def close_span():
        nonlocal current_type, start, end
        if current_type is not None:
            spans.append({
                "type": current_type,
                "start": start,
                "end": end,
                "text": utt[start:end],
            })
        current_type = None
        start = None
        end = None

    for label_id, (token_start, token_end) in zip(pred_ids, offsets):
        tag = id_to_slot[int(label_id)]

        if tag == "O":
            close_span()
            continue

        prefix, slot_type = tag.split("-", 1)

        if (
            prefix == "B"
            or current_type != slot_type
            or current_type is None
        ):
            close_span()
            current_type = slot_type
            start = token_start
            end = token_end
        else:
            end = token_end

    close_span()
    return spans


def span_key(span):
    return (
        span["type"],
        span["start"],
        span["end"],
    )


def compute_slot_micro_f1(predicted_spans, gold_spans):
    tp = fp = fn = 0

    for predictions, gold in zip(predicted_spans, gold_spans):
        pred_set = {span_key(s) for s in predictions}
        gold_set = {span_key(s) for s in gold}

        tp += len(pred_set & gold_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    denominator = 2 * tp + fp + fn
    f1 = 100 * (2 * tp) / denominator if denominator else 100.0

    return {
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


In [ ]:
def train_slot_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        slot_labels = batch["slot_labels"].to(device)

        # 1. Clear previous gradients
        optimizer.zero_grad()

        # 2. Forward pass
        logits = model(input_ids, attention_mask)

        # 3. Compute token-level loss
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            slot_labels.reshape(-1)
        )

        # 4. Backpropagation
        loss.backward()

        # 5. Stabilize training
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # 6. Update trainable parameters
        optimizer.step()

        valid_tokens = (slot_labels != SLOT_PAD_ID).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


In [ ]:
@torch.no_grad()
def evaluate_slot(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    all_predictions = []
    all_gold_spans = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        slot_labels = batch["slot_labels"].to(device)

        logits = model(input_ids, attention_mask)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            slot_labels.reshape(-1)
        )

        valid_tokens = (slot_labels != SLOT_PAD_ID).sum().item()

        total_loss += loss.item() * valid_tokens
        total_tokens += valid_tokens

        # Predicted BIO label IDs
        predictions = logits.argmax(dim=-1).cpu()

        for i in range(len(batch["utt"])):
            # Remove padding predictions
            seq_len = int(
                batch["attention_mask"][i].sum().item()
            )

            pred_ids = predictions[i, :seq_len].tolist()

            # Convert BIO tags to original-text spans
            pred_spans = decode_slot_spans(
                pred_ids,
                batch["offsets"][i],
                batch["utt"][i],
                id_to_slot
            )

            all_predictions.append(pred_spans)
            all_gold_spans.append(batch["gold_spans"][i])

    metrics = compute_slot_micro_f1(
        all_predictions,
        all_gold_spans
    )

    avg_loss = total_loss / max(total_tokens, 1)

    return avg_loss, metrics


In [ ]:
if RUN_SLOT_TRAINING:

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_f1": [],
        "epoch_time": []
    }

    best_val_f1 = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    training_start = time.time()

    for epoch in range(1, SLOT_EPOCH + 1):
        epoch_start = time.time()

        # Train
        train_loss = train_slot_epoch(
            slot_model,
            train_joint_loader,
            slot_optimizer,
            slot_criterion,
            device
        )

        # Validate
        val_loss, val_metrics = evaluate_slot(
            slot_model,
            val_joint_loader,
            slot_criterion,
            device
        )

        val_f1 = val_metrics["f1"]
        elapsed = time.time() - epoch_start

        # Record metrics
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)
        history["epoch_time"].append(elapsed)

        # Update learning rate based on validation slot F1.
        old_lr = slot_optimizer.param_groups[0]["lr"]
        slot_scheduler.step(val_f1)
        new_lr = slot_optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Slot F1: {val_f1:.2f}% | "
            f"Time: {elapsed:.1f}s | "
            f"LR: {new_lr:.2e}"
        )

        if new_lr < old_lr:
            print(f"Learning rate reduced: {old_lr:.2e} -> {new_lr:.2e}")

        # Save best checkpoint based on validation F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch
            epochs_without_improvement = 0

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": slot_model.state_dict(),
                    "val_f1": val_f1,
                    "val_loss": val_loss,
                    "use_backbone": USE_BACKBONE,
                    "pretraining_task": PRETRAINING_TASK,
                    "mask_ratio": MASK_RATIO,
                },
                BEST_SLOT_PATH
            )

            print(f"  -> Saved best checkpoint: {BEST_SLOT_PATH}")

        else:
            epochs_without_improvement += 1

        # Early stopping
        if epochs_without_improvement >= SLOT_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    total_training_time = time.time() - training_start

    print("\nTraining completed!")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation Slot F1: {best_val_f1:.2f}%")
    print(f"Checkpoint: {BEST_SLOT_PATH}")
    print(f"Total training time: {total_training_time:.1f}s")

else:
    print("Skipping slot filling training.")

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 4))
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Slot Extraction Training and Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(epochs, history["val_f1"], label="Validation Slot F1")
plt.xlabel("Epoch")
plt.ylabel("Slot Micro F1 (%)")
plt.title("Validation Slot F1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
checkpoint = torch.load(
    BEST_SLOT_PATH,
    map_location=device,
    weights_only=True
)

slot_model.load_state_dict(
    checkpoint["model_state_dict"]
)

slot_model.eval()

val_loss, val_metrics = evaluate_slot(
    slot_model,
    val_joint_loader,
    slot_criterion,
    device
)

print("Best checkpoint epoch:", checkpoint["epoch"])
print(f"Saved validation F1: {checkpoint['val_f1']:.2f}%")
print(f"Reloaded validation F1: {val_metrics['f1']:.2f}%")

In [ ]:
val_loss, val_metrics = evaluate_slot(
    slot_model,
    val_joint_loader,
    slot_criterion,
    device
)

print(f"Slot F1: {val_metrics['f1']:.2f}%")
print("TP:", val_metrics["tp"])
print("FP:", val_metrics["fp"])
print("FN:", val_metrics["fn"])

test_loss, test_metrics = evaluate_slot(
    slot_model,
    test_joint_loader,
    slot_criterion,
    device
)

print(f"Slot F1: {test_metrics['f1']:.2f}%")
print("TP:", test_metrics["tp"])
print("FP:", test_metrics["fp"])
print("FN:", test_metrics["fn"])

## Model Size

In [ ]:
run_params = {
    id(p): p
    for model in (backbone_model, intent_model, slot_model)
    for p in model.parameters()
}.values()

print(f"Pretraining task: {PRETRAINING_TASK}")
print(f"Total parameters: {sum(p.numel() for p in run_params):,}")